In [20]:
import os, importlib.util, shutil

!pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo transformers==5.2.0

cache_dirs = [
    "/root/.cache/torch_extensions",
    "/root/.cache/unsloth",
    "/content/unsloth_compiled_cache"
]
for d in cache_dirs:
    if os.path.exists(d):
        shutil.rmtree(d)
print("Caches cleared.")

Caches cleared.


In [21]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

In [22]:
max_seq_length = 2048

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3.5-2B",
    max_seq_length=max_seq_length,
    dtype=torch.bfloat16,
    load_in_4bit=True,        # quantized
    load_in_16bit=False,
    full_finetuning=False,    # LoRA only
    trust_remote_code=True,
)

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Qwen3_5 does not support SDPA - switching to fast eager.


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

In [23]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=False,
    random_state=3407,
    max_seq_length=max_seq_length,
    finetune_vision_layers=False,
)

# unwrap the tokenizer
tokenizer = tokenizer.tokenizer

# ensure pad token is defined
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

Unsloth: Making `model.base_model.model.model.language_model` require gradients


In [ ]:
train_dataset = load_dataset(
    "json",
    data_files={"train": "data/train.jsonl"},
    split="train"
)

SYSTEM_PROMPT = (
    "You are a protein localization database assistant. "
    "Always respond with a single valid JSON object and nothing else — "
    "no prose, no markdown, no code fences. "
    "The JSON must contain at minimum a \"primary_location\" field."
)

def convert_to_conversation(sample):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": sample["instruction"]},
        {"role": "assistant", "content": sample["response"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_set = train_dataset.map(
    convert_to_conversation,
    remove_columns=train_dataset.column_names
)

def tokenize_function(sample):
    return tokenizer(
        sample["text"],
        truncation=True,
        padding="max_length",
        max_length=max_seq_length,
    )

train_set = train_set.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)
print(train_set.column_names)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_set,
    tokenizer=tokenizer,
    packing=False,
    args=SFTConfig(
        max_seq_length=max_seq_length,
        per_device_train_batch_size=4,      # was 1 — use full A100 bandwidth
        gradient_accumulation_steps=4,      # effective batch = 16
        warmup_steps=50,
        max_steps=1250,                     # was 100 — now covers ~1 full epoch over 5k examples
        logging_steps=25,
        save_steps=250,
        output_dir="outputs_qwen35",
        optim="adamw_8bit",
        seed=3407,
        dataset_num_proc=4,
        remove_unused_columns=False,
        fp16=True,
        bf16=False,
        lr_scheduler_type="cosine",
        learning_rate=2e-4,
    )
)

In [27]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


Step,Training Loss
1,7.113893
2,6.826441
3,6.856631
4,6.293091
5,3.770592
6,1.696417
7,0.693288
8,0.802514
9,0.559313
10,0.665578


In [42]:
model.save_pretrained_merged(
    "/content/drive/MyDrive/locus.ai/merged_model",
    tokenizer,
    save_method="merged_16bit",
)

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 1 files from cache to `/content/drive/MyDrive/locus.ai/merged_model`:   0%|          | 0/1 [00:00<?, ?it/s]
Unsloth: Copying 1 files from cache to `/content/drive/MyDrive/locus.ai/merged_model`: 100%|██████████| 1/1 [00:48<00:00, 48.30s/it]


Successfully copied all 1 files from cache to `/content/drive/MyDrive/locus.ai/merged_model`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 11554.56it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:08<00:00, 68.32s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/locus.ai/merged_model`


In [ ]:
model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")